# DataDome: from a customer's fraud ticket to a better detection signal - analysis notebook
Independent case study based on public data, not affiliated with or endorsed by DataDome.
PUBLIC DATA = RBA login dataset (synthesized, Feb 2020 - Feb 2021) and web robot sessions (March 2018).
SIMULATED = a fictional anti-fraud platform with 12 fictional customers (seed 42). Outputs are short on purpose; the full
tables are in `data/output/`. Run `python run_all.py --skip-download` first.

In [ ]:
import json, duckdb, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'data' / 'output'
r = lambda n: pd.read_csv(OUT / f'{n}.csv')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 12)

## 1. Sources (read on 26 Nov 2026)

In [ ]:
f = json.loads((ROOT / 'data/public/sources_facts.json').read_text(encoding='utf-8'))
print(f['job']['title'], '|', f['job']['departments'], '| first published', f['job']['first_published'][:10])
print('SIRENE:', f['sirene']['name'], f['sirene']['siren'], '| accounts', f['sirene']['accounts_year'])
print('reference lists:', {k: v for k, v in f['reference_lists'].items() if k.endswith(('patterns', 'prefixes', 'ips'))})

## 2. RBA profile (PUBLIC DATA, synthesized values)

In [ ]:
r('rba_calibration').T

In [ ]:
r('rba_monthly')[['month', 'logins', 'attack_ip_share', 'ato_logins']].tail(6)   # no takeover label after Nov 2020

## 3. As-of signals and rules with a time split (train Feb-Aug 2020, test Sep-Nov 2020)

In [ ]:
r('rba_signal_rates')

In [ ]:
r('rba_rules').query("period == 'test'")[['rule', 'precision', 'recall', 'alerts_per_day', 'false_alerts_per_ato']]

In [ ]:
r('rba_topk')

Leakage check (SIMULATED, E7): a model that reads the account table extracted after the facts looks perfect; signals
known at login time with a time split give the honest number.

In [ ]:
r('e7_auc')[['model', 'evaluated_on', 'auc', 'precision_at_alert', 'recall_at_alert', 'alerts_per_day']]

## 4. Behavioural bot signals (PUBLIC DATA, March 2018, label as given by the dataset)

In [ ]:
r('webrobots_feature_auc').head(6)[['feature', 'auc', 'higher_for']]

In [ ]:
r('webrobots_models')

In [ ]:
r('lab_scores')[['visitor', 'kind', 'NUMBER_OF_REQUESTS', 'robot_score']]   # local lab, 127.0.0.1 only

## 5. Monitoring of the simulated platform (SIMULATED)

In [ ]:
con = duckdb.connect()
con.sql(f"""SELECT customer_id, COUNT(*) AS raw_events, COUNT(DISTINCT event_id) AS unique_events
           FROM read_parquet('{(ROOT / 'data/sim/login_events/*/*.parquet').as_posix()}', hive_partitioning = true)
           GROUP BY 1 ORDER BY 1""").df()

In [ ]:
r('alerts')[['week', 'customer_id', 'status']]

## 6. The 8 events

In [ ]:
r('tests')[['test', 'passed']]

In [ ]:
print(r('e1_summary').T, '
', r('e1_ip_rate').T)

In [ ]:
r('e3_summary').T

In [ ]:
r('e4_crawlers').head(3)

In [ ]:
r('e5_dedup_effect').tail(4)

In [ ]:
r('e6_simpson')

## 7. Customer case T07

In [ ]:
r('e2_summary').T

In [ ]:
print((ROOT / 'docs/customer_case_report_T07.md').read_text(encoding='utf-8')[:900])

## 8. Signal proposals, measured offline (SIMULATED)

In [ ]:
r('signal_offline')[['signal', 'alerts_per_day', 'precision', 'recall', 'baseline_alerts_per_day']]